# EcoSentinel — Phase 6: PM2.5 Forecasting
## LSTM Training + TimesFM 2.0 Zero-Shot Comparison

**What this notebook does, top to bottom:**
1. Clones the EcoSentinel project and installs dependencies
2. Fetches 1 year of PM2.5 + weather data for 4 cities via Open-Meteo
3. Builds a sliding-window dataset (30 days → predict day 31)
4. Trains a stacked LSTM from scratch in PyTorch
5. Evaluates against persistence + rolling-average baselines
6. Optionally compares against Google TimesFM 2.0 zero-shot
7. Downloads the trained model so you can drop it in `data/model/`

**How to use this notebook:**
- Run cells **in order**, top to bottom. Never skip a cell.
- Each cell has a comment at the top saying what it does and roughly how long it takes.
- Click the ▶ button on the left of each cell, or press **Shift+Enter**.
- A `[*]` next to a cell means it is still running. Wait for it to become `[1]`, `[2]`, etc.

**Recommended runtime:** Runtime → Change runtime type → T4 GPU → Save

In [ ]:
# ── Cell 1: Clone the repo and install dependencies ───────────────────────────
# Takes ~1 minute on first run. If you see 'already exists', skip to Cell 2.

!git clone https://github.com/maahipatel05/ecosentinel.git
%cd ecosentinel

# Core dependencies (torch is the big one — ~800MB, takes ~1 min)
!pip install torch httpx numpy --quiet

# Uncomment the line below to also run the TimesFM zero-shot comparison.
# Warning: downloads an additional ~800MB model checkpoint from HuggingFace.
# !pip install timesfm --quiet

print("\n✅ Installation complete")

In [ ]:
# ── Cell 2: Imports and configuration ────────────────────────────────────────
# All constants live here. Change TRAIN_START/TRAIN_END to experiment
# with different training windows. Takes <5 seconds.

import asyncio
import json
import sys
import time
from pathlib import Path

import httpx
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

# Import the shared model definition from src/forecast.py
# (PM25LSTM, normalize, denormalize_pm25 — defined once, used in both training and server)
sys.path.insert(0, "src")
from forecast import (
    PM25LSTM,
    N_FEATURES,
    SEQ_LEN,
    HIDDEN1,
    HIDDEN2,
    DROPOUT,
    normalize,
    denormalize_pm25,
    MODEL_DIR,
    MODEL_PATH,
    SCALER_PATH,
)

# ── Training hyperparameters ─────────────────────────────────────────────────
TRAIN_START = "2024-01-01"  # full year of training data
TRAIN_END   = "2024-12-31"
BATCH_SIZE  = 32
MAX_EPOCHS  = 200
PATIENCE    = 15   # stop early if val loss doesn't improve for 15 epochs
LR          = 1e-3
TRAIN_FRAC  = 0.80
VAL_FRAC    = 0.10  # remaining 10% = test

# ── Four training cities (chosen for diversity and data quality) ──────────────
# Delhi: chronic winter smog | LA: wildfire-driven spikes
# Seoul: East Asian pollution outflow | Krakow: coal heating season
CITIES = {
    "Delhi":       (28.6517,   77.2219),
    "Los Angeles": (34.0522, -118.2437),
    "Seoul":       (37.5665,  126.9780),
    "Krakow":      (50.0647,   19.9450),
}

AIR_QUALITY_URL     = "https://air-quality-api.open-meteo.com/v1/air-quality"
WEATHER_ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"Sequence length : {SEQ_LEN} days  →  predict day {SEQ_LEN + 1}")
print(f"Features        : PM2.5, wind_speed_10m, temperature_2m  ({N_FEATURES} total)")
print(f"Training window : {TRAIN_START} to {TRAIN_END}")
print(f"Cities          : {', '.join(CITIES)}")
print("\n✅ Configuration ready")

---
## Step 1 — Fetch Training Data

We pull 1 year of data from two Open-Meteo APIs:

- **Air Quality API** → hourly PM2.5 values → averaged to daily means
- **Weather Archive API** → hourly wind speed + temperature → averaged to daily means

We only keep days where **all three values** are present (no partial days).
1 year × 4 cities = roughly 1,400 clean daily rows total.

The `async/await` pattern sends both API calls for each city **at the same time**
instead of one after the other — that's why fetching all 4 cities takes ~20 seconds
instead of ~80 seconds.

In [ ]:
# ── Cell 3: Data fetching functions ──────────────────────────────────────────
# Defines the fetch logic. Running this cell does NOT make any API calls yet.
# That happens in Cell 4 below.

async def _fetch_city_data(city: str, lat: float, lon: float) -> list:
    """Fetch daily PM2.5 + wind + temperature for TRAIN_START → TRAIN_END."""
    async with httpx.AsyncClient(timeout=60) as client:
        r_aq, r_wx = await asyncio.gather(
            client.get(AIR_QUALITY_URL, params={
                "latitude": lat, "longitude": lon,
                "hourly": "pm2_5",
                "start_date": TRAIN_START, "end_date": TRAIN_END,
                "timezone": "UTC",
            }),
            client.get(WEATHER_ARCHIVE_URL, params={
                "latitude": lat, "longitude": lon,
                "hourly": "temperature_2m,wind_speed_10m",
                "start_date": TRAIN_START, "end_date": TRAIN_END,
                "timezone": "UTC",
            }),
        )

    if r_aq.status_code != 200 or r_wx.status_code != 200:
        raise RuntimeError(f"{city}: AQ={r_aq.status_code} WX={r_wx.status_code}")

    def _hourly_to_daily(times, values):
        by_date = {}
        for t, v in zip(times, values):
            if v is None or v < 0:
                continue
            by_date.setdefault(t[:10], []).append(float(v))
        return {d: float(np.mean(vs)) for d, vs in by_date.items()}

    aq_h   = r_aq.json()["hourly"]
    wx_h   = r_wx.json()["hourly"]
    pm25_d = _hourly_to_daily(aq_h["time"], aq_h["pm2_5"])
    wind_d = _hourly_to_daily(wx_h["time"], wx_h["wind_speed_10m"])
    temp_d = _hourly_to_daily(wx_h["time"], wx_h["temperature_2m"])

    common = sorted(set(pm25_d) & set(wind_d) & set(temp_d))
    return [{"date": d, "pm25": pm25_d[d], "wind": wind_d[d], "temp": temp_d[d]}
            for d in common]


async def fetch_all_cities() -> dict:
    """Fetch data for all four cities sequentially with polite pacing."""
    all_data = {}
    for city, (lat, lon) in CITIES.items():
        print(f"  Fetching {city}...", end=" ", flush=True)
        t0 = time.time()
        rows = await _fetch_city_data(city, lat, lon)
        print(f"{len(rows)} days ({time.time() - t0:.1f}s)")
        all_data[city] = rows
        await asyncio.sleep(0.5)
    return all_data

print("✅ Fetch functions defined")

In [ ]:
# ── Cell 4: Actually fetch the data ──────────────────────────────────────────
# Makes real API calls. Takes ~20-30 seconds.
# You should see each city print as it completes.

print("Fetching 1 year of training data from Open-Meteo...")
city_data = await fetch_all_cities()

total = sum(len(v) for v in city_data.values())
print(f"\n✅ Total: {total} daily rows across {len(city_data)} cities")

# Quick sanity check — print a few rows from Delhi
print("\nSample rows from Delhi (first 3 days):")
for row in city_data["Delhi"][:3]:
    print(f"  {row['date']}  PM2.5={row['pm25']:.1f}  wind={row['wind']:.1f}  temp={row['temp']:.1f}")

---
## Step 2 — Build the Dataset

### Sliding windows

We convert the daily rows into overlapping 30-day windows. Each window is one
training example:

```
Input  X[i] : days 1–30  → shape (30, 3)   ← 30 days × 3 features
Target y[i] : day 31's PM2.5               ← single number

Input  X[i+1]: days 2–31 → shape (30, 3)
Target y[i+1]: day 32's PM2.5
... and so on
```

From ~365 days per city × 4 cities we get roughly **1,340 training examples total**.
Windows from different cities never overlap — Delhi's last window doesn't bleed into LA's first.

### Normalization

PM2.5 can be 500 µg/m³ in Delhi. Temperature might be 15°C.
If we feed raw values, PM2.5 completely drowns out temperature mathematically.
Min-max normalization rescales every feature to [0, 1]:

```
normalized = (value - feature_min) / (feature_max - feature_min)
```

**Critical rule:** we fit the scaler on the **training set only**.
If we used the full dataset's min/max, we would leak test data into training — 
the model would secretly know how extreme the test set's values are.

In [ ]:
# ── Cell 5: Build sequences, split, and fit the scaler ───────────────────────
# Pure CPU work — takes <5 seconds.

def make_sequences(city_data: dict):
    """Convert per-city daily rows into overlapping (X, y) sliding windows."""
    X_all, y_all = [], []
    for city, rows in city_data.items():
        values = np.array(
            [[r["pm25"], r["wind"], r["temp"]] for r in rows],
            dtype=np.float32,
        )
        for i in range(len(values) - SEQ_LEN):
            X_all.append(values[i : i + SEQ_LEN])   # 30 days, 3 features
            y_all.append(values[i + SEQ_LEN, 0])     # next day's PM2.5 only
    return np.array(X_all), np.array(y_all, dtype=np.float32)


def split_data(X, y):
    n     = len(X)
    n_tr  = int(n * TRAIN_FRAC)
    n_val = int(n * VAL_FRAC)
    return (
        X[:n_tr],            y[:n_tr],
        X[n_tr:n_tr+n_val],  y[n_tr:n_tr+n_val],
        X[n_tr+n_val:],      y[n_tr+n_val:],
    )


def fit_scaler(X_train: np.ndarray) -> dict:
    """Compute per-feature min/max from training windows only."""
    flat = X_train.reshape(-1, N_FEATURES)  # flatten all time steps
    return {
        "features": ["pm25", "wind_speed_10m", "temperature_2m"],
        "min": flat.min(axis=0).tolist(),
        "max": flat.max(axis=0).tolist(),
    }


# Build and split
X, y = make_sequences(city_data)
X_tr, y_tr, X_va, y_va, X_te, y_te = split_data(X, y)

# Fit scaler on training data only
scaler = fit_scaler(X_tr)
SCALER_PATH.write_text(json.dumps(scaler, indent=2))

print(f"Total sequences : {len(X)}")
print(f"  Train         : {len(X_tr)}")
print(f"  Validation    : {len(X_va)}")
print(f"  Test          : {len(X_te)}")
print(f"\nScaler fitted and saved → {SCALER_PATH}")
print(f"PM2.5 range in training: {scaler['min'][0]:.1f} – {scaler['max'][0]:.1f} µg/m³")

---
## Step 3 — Train the LSTM

The model is a **stacked LSTM** — two LSTM layers feeding into one final linear layer:

```
Input:  (batch=32, seq=30, features=3)
  ↓
LSTM Layer 1   (64 hidden units)   ← learns short-term patterns
  ↓
LSTM Layer 2   (32 hidden units)   ← learns longer-range patterns
  ↓
Dropout(0.2)                       ← prevents memorisation
  ↓
Linear(32 → 1)                     ← collapses to a single number
  ↓
Output: tomorrow's normalised PM2.5
```

**Early stopping**: we save the model whenever validation loss improves.
If val loss stops improving for 15 consecutive epochs, we stop and restore
the best saved weights. This prevents the model from overfitting.

In [ ]:
# ── Cell 6: PyTorch Dataset wrapper ──────────────────────────────────────────
# PyTorch's DataLoader needs a Dataset object that knows how to:
#   1. Tell it how many examples exist (__len__)
#   2. Return example #i when asked (__getitem__)
# DataLoader then handles batching and shuffling automatically.

class PM25Dataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.x = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

print("✅ PM25Dataset class defined")

In [ ]:
# ── Cell 7: Training loop ─────────────────────────────────────────────────────
# Takes ~3-5 minutes on CPU, ~30 seconds on T4 GPU.
# Watch the Val Loss column — when it stops decreasing, the model has learned
# as much as it can. Early stopping kicks in automatically.

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

# ── Normalise inputs and targets ──────────────────────────────────────────────
# Targets are PM2.5 values (feature index 0) — normalise them too
# so the model's output lives in [0, 1] space
pm25_min = scaler["min"][0]
pm25_max = scaler["max"][0]

X_tr_n = normalize(X_tr.reshape(-1, N_FEATURES), scaler).reshape(X_tr.shape)
X_va_n = normalize(X_va.reshape(-1, N_FEATURES), scaler).reshape(X_va.shape)

y_tr_n = (y_tr - pm25_min) / (pm25_max - pm25_min + 1e-8)
y_va_n = (y_va - pm25_min) / (pm25_max - pm25_min + 1e-8)

tr_loader = DataLoader(PM25Dataset(X_tr_n, y_tr_n), batch_size=BATCH_SIZE, shuffle=True)
va_loader = DataLoader(PM25Dataset(X_va_n, y_va_n), batch_size=BATCH_SIZE)

model     = PM25LSTM().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

best_val_loss  = float("inf")
patience_count = 0

print(f"\n{'Epoch':>6}  {'Train Loss':>12}  {'Val Loss':>12}  Status")
print("-" * 55)

for epoch in range(1, MAX_EPOCHS + 1):
    # ── Train ──────────────────────────────────────────────────────────────────
    model.train()
    tr_losses = []
    for xb, yb in tr_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        tr_losses.append(loss.item())
    tr_loss = float(np.mean(tr_losses))

    # ── Validate ───────────────────────────────────────────────────────────────
    model.eval()
    va_losses = []
    with torch.no_grad():
        for xb, yb in va_loader:
            xb, yb = xb.to(device), yb.to(device)
            va_losses.append(criterion(model(xb), yb).item())
    va_loss = float(np.mean(va_losses))

    if va_loss < best_val_loss:
        best_val_loss = va_loss
        torch.save(model.state_dict(), MODEL_PATH)
        patience_count = 0
        status = "✓ saved"
    else:
        patience_count += 1
        status = f"({patience_count}/{PATIENCE})"

    if epoch % 10 == 0 or epoch <= 3 or patience_count >= PATIENCE:
        print(f"{epoch:>6}  {tr_loss:>12.5f}  {va_loss:>12.5f}  {status}")

    if patience_count >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}.")
        break

# Restore the best weights (from the epoch we saved)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))
model.eval()

print(f"\n✅ Training complete — best val loss: {best_val_loss:.5f}")
print(f"   Model saved → {MODEL_PATH}")

---
## Step 4 — Evaluate

We compare three models against two baselines on the **held-out test set**
(the 10% of data the model has never seen).

| Metric | What it measures | Units |
|---|---|---|
| **MAE** | Average absolute error | µg/m³ |
| **RMSE** | Penalises large errors more than MAE | µg/m³ |
| **MAPE** | Scale-independent error (fair across cities) | % |

**Baselines we compare against:**
- **Persistence**: predict tomorrow = today. Simple but hard to beat.
- **7-day rolling average**: predict tomorrow = mean of last 7 days.

A model is only meaningful if it beats both baselines.

In [ ]:
# ── Cell 8: Evaluation functions + results ────────────────────────────────────
# Pure CPU — takes <5 seconds.

def compute_metrics(y_true, y_pred, label):
    y_true = np.array(y_true, dtype=np.float32)
    y_pred = np.clip(np.array(y_pred, dtype=np.float32), 0, None)
    mae  = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    mape = float(np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100)
    return {"model": label, "mae": mae, "rmse": rmse, "mape": mape}


# Persistence baseline: predict today's value (last day of input window)
preds_persistence = X_te[:, -1, 0]  # last day's raw PM2.5

# 7-day rolling average baseline
preds_rolling = X_te[:, -7:, 0].mean(axis=1)

# LSTM predictions (denormalize from [0,1] back to µg/m³)
X_te_n = normalize(X_te.reshape(-1, N_FEATURES), scaler).reshape(X_te.shape)
with torch.no_grad():
    preds_norm = model(torch.tensor(X_te_n, dtype=torch.float32).to(device))
    preds_norm = preds_norm.cpu().numpy()
preds_lstm = np.array([denormalize_pm25(float(p), scaler) for p in preds_norm])

# Compute metrics for all three
results = [
    compute_metrics(y_te, preds_persistence, "Persistence (baseline)"),
    compute_metrics(y_te, preds_rolling,     "7-day rolling avg (baseline)"),
    compute_metrics(y_te, preds_lstm,         "LSTM (ours)"),
]

# Print results table
print(f"\n{'Model':<34}  {'MAE':>8}  {'RMSE':>8}  {'MAPE':>8}")
print("-" * 64)
for r in results:
    print(f"{r['model']:<34}  {r['mae']:>7.2f}   {r['rmse']:>7.2f}   {r['mape']:>6.1f}%")

lstm_r = results[2]
pers_r = results[0]
improvement = (pers_r["mae"] - lstm_r["mae"]) / pers_r["mae"] * 100
print(f"\n→ LSTM vs Persistence: {improvement:.1f}% lower MAE")

---
## Step 5 — TimesFM 2.0 Zero-Shot Comparison (Optional)

This section runs Google's **TimesFM 2.0** — a foundation model pre-trained on
100 billion time points — and compares it against our LSTM on the same test set.

**Zero-shot** means TimesFM was never trained on PM2.5 data at all.
It forecasts purely based on patterns it learned from completely different domains
(web traffic, retail sales, energy grids, etc.).

**This section only runs if you uncommented `!pip install timesfm` in Cell 1.**
If timesfm is not installed, this cell prints a clear message and skips safely.

The first run downloads ~800MB from HuggingFace and caches it —
subsequent runs are instant.

In [ ]:
# ── Cell 9: TimesFM 2.0 zero-shot ─────────────────────────────────────────────
# Only runs if timesfm is installed. First run: ~3-5 minutes (model download).
# Re-runs: ~1-2 minutes (inference only).

try:
    import timesfm
    _TIMESFM_AVAILABLE = True
except ImportError:
    _TIMESFM_AVAILABLE = False

if not _TIMESFM_AVAILABLE:
    print("TimesFM not installed — skipping zero-shot comparison.")
    print("To run this section: restart runtime, uncomment the timesfm line in Cell 1, re-run all cells.")
else:
    print("Loading google/timesfm-2.0-200m-pytorch from HuggingFace...")
    print("(First run downloads ~800MB and takes ~3-5 minutes)")

    try:
        tfm = timesfm.TimesFm(
            hparams=timesfm.TimesFmHparams(
                backend="pytorch",
                per_core_batch_size=32,
                horizon_len=1,
            ),
            checkpoint=timesfm.TimesFmCheckpoint(
                huggingface_repo_id="google/timesfm-2.0-200m-pytorch",
            ),
        )

        # Feed raw (un-normalised) PM2.5 history as context
        # TimesFM is univariate — it only sees PM2.5, not wind or temperature
        # This is the whole point of zero-shot: no feature engineering, no training
        contexts = [X_te[i, :, 0].tolist() for i in range(len(X_te))]

        print(f"Running zero-shot inference on {len(contexts)} test sequences...")
        forecasts, _ = tfm.forecast(
            inputs=contexts,
            freq=[0] * len(contexts),  # 0 = daily/high frequency
        )
        preds_timesfm = np.clip([float(f[0]) for f in forecasts], 0, None)

        tfm_metrics = compute_metrics(y_te, preds_timesfm, "TimesFM 2.0 (zero-shot)")
        results.append(tfm_metrics)

        print(f"\n✅ TimesFM complete")
        print(f"\n{'Model':<34}  {'MAE':>8}  {'RMSE':>8}  {'MAPE':>8}")
        print("-" * 64)
        for r in results:
            print(f"{r['model']:<34}  {r['mae']:>7.2f}   {r['rmse']:>7.2f}   {r['mape']:>6.1f}%")

    except Exception as e:
        print(f"TimesFM inference failed: {e}")
        print("Check https://github.com/google-research/timesfm for any API changes.")

---
## Step 6 — Download the Trained Model

Colab's file system is **temporary** — it disappears when the session ends.
You need to download the model files to your computer before closing Colab.

After downloading:
1. Move both files into `ecosentinel/data/model/` on your local machine
2. The `get_forecast` MCP tool will automatically load them at server startup

In [ ]:
# ── Cell 10: Verify files exist, then download ────────────────────────────────
# Your browser will automatically download both files.
# If nothing happens, check your browser's download settings.

import os

model_size  = os.path.getsize(MODEL_PATH)  / 1024
scaler_size = os.path.getsize(SCALER_PATH) / 1024
print(f"lstm_pm25.pt  : {model_size:.1f} KB")
print(f"scaler.json   : {scaler_size:.1f} KB")
print()

# Resume line — fill in the numbers after you run this
lstm_r = next(r for r in results if r["model"] == "LSTM (ours)")
pers_r = next(r for r in results if "Persistence" in r["model"])
improv = (pers_r["mae"] - lstm_r["mae"]) / pers_r["mae"] * 100
print("── Resume line ──────────────────────────────────────────────")
print(f"  LSTM MAE = {lstm_r['mae']:.1f} µg/m³ ({improv:.0f}% below persistence baseline)")
print(f"  Trained on 1 year of Open-Meteo CAMS data across 4 global cities")
print()

# Download to your local machine
from google.colab import files
files.download(str(MODEL_PATH))
files.download(str(SCALER_PATH))
print("✅ Files downloading — check your browser's downloads folder.")
print("   Move them to: ecosentinel/data/model/ on your local machine.")